In [1]:
import numpy as np
import pandas as pd
import Soil_Carbon_Model as md
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit 
import os
import openpyxl
from pandas import *
%matplotlib inline

In [2]:
import pylab
params = {'backend': 'ps',
          'axes.labelsize': 20,
          'legend.fontsize': 20,
          'legend.handlelength': 2.0,
          'legend.borderaxespad': 0,
          'xtick.labelsize': 20,
          'ytick.labelsize': 20,
          'font.family': 'serif',
          'font.size': 20,
          'font.weight': 'normal',
          #'font.serif': ['Palatino'],
          'ps.usedistiller': 'xpdf',
          'text.usetex': False,
         'agg.path.chunksize' : 1000}
pylab.rcParams.update(params)

In [3]:
dt = 1 / 24
nyears = 2000
ndays = nyears * 365
nsim = ndays * 24
nnpp = ndays * 24

#Rainfall from Poisson
rain = md.poisson_process(n = int(nsim), alpha = 15.12/1000, lbd = (0.17 * dt))

tsim = np.linspace(1, nnpp, int(nnpp)) # Hour

days = np.arange(365)
annual_avg_npp = 492.8

# Half sine wave shape over 365 days
shape = np.sin((np.pi*days)/365)                       # sinB(x-h) + K; B = pi/365, h = 0, K = 0

# Normalize so that the sum equals the annual NPP
daily_npp = annual_avg_npp * (shape / np.sum(shape))
hourly_npp = np.repeat(daily_npp, 24)

npp = np.tile(hourly_npp,nyears)

In [4]:
print("Maximum Value is:", max(npp))
print("Length is:", len(npp))
print("Average value is:", np.mean(npp))

Maximum Value is: 2.1207836723891114
Length is: 17520000
Average value is: 1.35013698630137


In [5]:
total_rainfall = rain.sum()
print(f"Total rainfall over 450 years = {total_rainfall:.3f} meters")

RPY = (total_rainfall / nyears)*1000         # average rainfall per year
print(f"Average rainfall per year = {RPY:.3f} mm/year")

Total rainfall over 450 years = 1872.336 meters
Average rainfall per year = 936.168 mm/year


In [6]:
PET = 1260.9
AI = PET/RPY
print(AI)

"""
def hill_constrained(R, KA, n):
    R_max = 4.18

    numerator = R**n
    denominator = KA**n + R**n
    norm_factor = R_max**n / (KA**n + R_max**n)
    TI = (numerator / denominator) / norm_factor

    # Clip to ensure TI does not exceed 1
    return (np.minimum(TI, 1.0))

TI = hill_constrained(AI, 3.45, 2.28)
print(TI)
"""

# Final best-fit parameters from your latest run
KA_final = 9.0132
n_final = 1.7444
R_max = 3.8  # Max AI used for normalization

def compute_TI(AI):
    AI = np.asarray(AI)  # Accept scalars or arrays
    numerator = AI**n_final
    denominator = KA_final**n_final + AI**n_final
    norm_factor = R_max**n_final / (KA_final**n_final + R_max**n_final)
    TI = (numerator / denominator) / norm_factor
    return np.minimum(TI, 1.0)

TI = compute_TI(AI)
print("TI =", TI)

1.3468733058664852
TI = 0.19305775065038533


In [7]:
#Hydraulic parameterization for dynamic porosity -------------------------------
alpha1 = 580  #hydraulic conductivity shape parameter
beta1 = 6     #hydraulic conductivity shape parameter
alpha2 = 580  #porosity shape parameter
beta2 = 6     #porosity shape parameter
kstex = 0.1   #m/d,
kstot = 1     #m/d
phitex = 0.50  
phitot = 0.60  

In [8]:
# Simulation for 2150 years at dynamic porosity
sim = md.run_julia(rain, npp, kstex, kstot, phitex, phitot, alpha1, beta1, alpha2, beta2, TI)
sim_year = sim.pivot_table(values=['s', 'phi', 'ksstr', 'P', 'DOC', 'A', 'M', 'B',
                                   'Fa', 'Fpa','Flb','Fl', 'Flm', 'Fpl', 'Fbm','Fld', 'Fal',
                                   'lk','npp_', 'L','Fi', 'Flpom', 'decomp'],
                           index='Years')

print(sim_year)

100%|██████████| 17520000/17520000 [49:45<00:00, 5868.25it/s]  


                 A           B         DOC        Fa       Fal       Fbm  \
Years                                                                      
1       935.329130  144.910292  281.358198  0.219671  0.007030  0.969659   
2      1002.411103  122.733331  181.415090  0.233157  0.007251  0.821314   
3      1067.593336   69.971970  199.538793  0.251580  0.006954  0.468245   
4      1125.667635   51.463302  233.826170  0.266551  0.006763  0.344377   
5      1176.294008   49.162197  255.375619  0.276556  0.006929  0.328969   
...            ...         ...         ...       ...       ...       ...   
1996   1731.384507   72.807271  247.545482  0.399933  0.011175  0.487188   
1997   1731.782585   74.726965  255.865653  0.410098  0.011535  0.500040   
1998   1731.411599   90.036308  235.852224  0.386445  0.011333  0.602465   
1999   1731.079126   90.499320  211.413521  0.391318  0.011490  0.605586   
2000   1731.184266   79.661496  224.564414  0.409694  0.011699  0.533071   

           

In [11]:
sim_year.to_csv(r"C:\Users\rishabh.singh\OneDrive - Texas A&M AgriLife\Desktop\Slope\Bhattacharya\BC.csv", index= True, header = True)

sim_year.to_csv(r"C:\Users\rishabh.singh\OneDrive - Texas A&M AgriLife\Desktop\Slope\Results\BC.csv", index= True, header = True)